In [4]:
import pandas as pd 
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv
import math 

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
OPINET_API_KEY_GROUP = ['OPINET_API_KEY_1', 'OPINET_API_KEY_2', 'OPINET_API_KEY_3', 
                        'OPINET_API_KEY_4', 'OPINET_API_KEY_5', 'OPINET_API_KEY_6' ] 
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT') 
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')


#2. duckdb를 통한 s3 읽기 설정
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{MINIO_ENDPOINT}';")
con.execute(f"SET s3_access_key_id='{MINIO_ACCESS_KEY}';")
con.execute(f"SET s3_secret_access_key='{MINIO_SECRET_KEY}';")
con.execute("SET s3_url_style='path'; SET s3_use_ssl='false';")


print("✅ DuckDB의 MinIO 접속 준비 완료!")



✅ DuckDB의 MinIO 접속 준비 완료!


In [ ]:
# A1. 기초적인 Pearson Correlation 계산
df = con.sql(
    """
    with base as (
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    and part_dt >= '20260101'
    group by 1
    )
    select 
    CORR(gasoline_prc , disel_prc) as pearson_correlation
    from base 
"""
).df()

display(df)

,pearson_correlation
0,0.99153


In [5]:
# A2. 연도별 Pearson Correlation 계산
import matplotlib

df = con.sql(
    """
    with base as (
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    group by 1
    )
    select 
    substring( cast(part_dt AS STRING), 1,4) as yyyy
    , CORR(gasoline_prc , disel_prc) as pearson_correlation
    from base 
    group by 1
    order by 1
"""
).df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
# A1. 기초적인 Pearson Correlation 계산
df = con.sql(
    """
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    and part_dt >= '20200101'
    group by 1
    """
).df()

display(df)

,part_dt,gasoline_prc,disel_prc
0,20200229,1525.03,1344.88
1,20200104,1561.87,1393.82
2,20200326,1418.18,1224.52
3,20200214,1545.01,1370.38
4,20210313,1509.97,1309.59
...,...,...,...
2339,20260526,2011.22,2005.64
2340,20260527,2011.09,2005.31
2341,20260529,2010.81,2005.45
2342,20260530,2010.77,2005.26


In [ ]:
#A3. pandas로 corr 계산하기
import pandas as pd

corr = df['gasoline_prc'].corr(df['disel_prc'])
display(corr)

0.9925729779446998

In [23]:
#A4. 연도별 corr 계산하기
import pandas as pd 
df['yyyy'] = df['part_dt'].astype(str).str[:4]

yearly_corr = df.groupby('yyyy').apply(lambda y: df['disel_prc'].corr(df['gasoline_prc']))

display(yearly_corr)

C:\Users\B450M\AppData\Local\Temp\ipykernel_7104\2436207002.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  yearly_corr = df.groupby('yyyy').apply(lambda y: df['disel_prc'].corr(df['gasoline_prc']))


yyyy
2020    0.919215
2021    0.919215
2022    0.919215
2023    0.919215
2024    0.919215
2025    0.919215
2026    0.919215
dtype: float64